# Optimize SQL Prompts: Context Engineering for Better Query Results

Previously, we tested the below prompt:

>I am working with a dataset that contains information about the Red30 Tech company's product sales online. 
>I want to create a SQL query that calculates the total sales by state.

While it returns a SQL code, it was generic and cannot be used out of the box as it missing critical context. 

In this example we will see how we can improve the LLM's response with providing additional context.

Let's start by importing the openai and os libraries:

In [8]:
from openai import OpenAI
import os

In [9]:
openai_api_key = os.environ.get("OPENAI_API_KEY")
#print(openai_api_key)

In [10]:
client = OpenAI(api_key= openai_api_key)

Next, we define the number of tokens and temperature:

In [11]:
temperature = 0
max_tokens = 5000

Next, let's add to the prompt additional information with the table name and column headers:

In [14]:
content_prompt = """
I am working with a dataset that contains information about the national bridge inventory data for Alaska. 

I want to create a SQL query that calculates the total number of bridges in Alaska.

The table name is nbi_2025_ak which stands for "National Bridge Inventory 2025 Alaska".
Note that this table only contains bridge data for Alaska (STATE_CODE_001 is always 2),
so we do not need to filter by state.

All the column headers are listed herein:
STATE_CODE_001
STRUCTURE_NUMBER_008
RECORD_TYPE_005A
ROUTE_PREFIX_005B
SERVICE_LEVEL_005C
ROUTE_NUMBER_005D
DIRECTION_005E
HIGHWAY_DISTRICT_002
COUNTY_CODE_003
PLACE_CODE_004
FEATURES_DESC_006A
CRITICAL_FACILITY_006B
FACILITY_CARRIED_007
LOCATION_009
MIN_VERT_CLR_010
KILOPOINT_011
BASE_HWY_NETWORK_012
LRS_INV_ROUTE_013A
SUBROUTE_NO_013B
LAT_016
LONG_017
DETOUR_KILOS_019
TOLL_020
MAINTENANCE_021
OWNER_022
FUNCTIONAL_CLASS_026
YEAR_BUILT_027
TRAFFIC_LANES_ON_028A
TRAFFIC_LANES_UND_028B
ADT_029
YEAR_ADT_030
DESIGN_LOAD_031
APPR_WIDTH_MT_032
MEDIAN_CODE_033
DEGREES_SKEW_034
STRUCTURE_FLARED_035
RAILINGS_036A
TRANSITIONS_036B
APPR_RAIL_036C
APPR_RAIL_END_036D
HISTORY_037
NAVIGATION_038
NAV_VERT_CLR_MT_039
NAV_HORR_CLR_MT_040
OPEN_CLOSED_POSTED_041
SERVICE_ON_042A
SERVICE_UND_042B
STRUCTURE_KIND_043A
STRUCTURE_TYPE_043B
APPR_KIND_044A
APPR_TYPE_044B
MAIN_UNIT_SPANS_045
APPR_SPANS_046
HORR_CLR_MT_047
MAX_SPAN_LEN_MT_048
STRUCTURE_LEN_MT_049
LEFT_CURB_MT_050A
RIGHT_CURB_MT_050B
ROADWAY_WIDTH_MT_051
DECK_WIDTH_MT_052
VERT_CLR_OVER_MT_053
VERT_CLR_UND_REF_054A
VERT_CLR_UND_054B
LAT_UND_REF_055A
LAT_UND_MT_055B
LEFT_LAT_UND_MT_056
DECK_COND_058
SUPERSTRUCTURE_COND_059
SUBSTRUCTURE_COND_060
CHANNEL_COND_061
CULVERT_COND_062
OPR_RATING_METH_063
OPERATING_RATING_064
INV_RATING_METH_065
INVENTORY_RATING_066
STRUCTURAL_EVAL_067
DECK_GEOMETRY_EVAL_068
UNDCLRENCE_EVAL_069
POSTING_EVAL_070
WATERWAY_EVAL_071
APPR_ROAD_EVAL_072
WORK_PROPOSED_075A
WORK_DONE_BY_075B
IMP_LEN_MT_076
DATE_OF_INSPECT_090
INSPECT_FREQ_MONTHS_091
FRACTURE_092A
UNDWATER_LOOK_SEE_092B
SPEC_INSPECT_092C
FRACTURE_LAST_DATE_093A
UNDWATER_LAST_DATE_093B
SPEC_LAST_DATE_093C
BRIDGE_IMP_COST_094
ROADWAY_IMP_COST_095
TOTAL_IMP_COST_096
YEAR_OF_IMP_097
OTHER_STATE_CODE_098A
OTHER_STATE_PCNT_098B
OTHR_STATE_STRUC_NO_099
STRAHNET_HIGHWAY_100
PARALLEL_STRUCTURE_101
TRAFFIC_DIRECTION_102
TEMP_STRUCTURE_103
HIGHWAY_SYSTEM_104
FEDERAL_LANDS_105
YEAR_RECONSTRUCTED_106
DECK_STRUCTURE_TYPE_107
SURFACE_TYPE_108A
MEMBRANE_TYPE_108B
DECK_PROTECTION_108C
PERCENT_ADT_TRUCK_109
NATIONAL_NETWORK_110
PIER_PROTECTION_111
BRIDGE_LEN_IND_112
SCOUR_CRITICAL_113
FUTURE_ADT_114
YEAR_OF_FUTURE_ADT_115
MIN_NAV_CLR_MT_116
FED_AGENCY
SUBMITTED_BY
BRIDGE_CONDITION
LOWEST_RATING
DECK_AREA
"""

Let's now use the GPT5 model to send the prompt:

In [15]:
response = client.chat.completions.create(
    model="gpt-5",
    messages=[{"role": "user", "content": content_prompt}],
    max_completion_tokens=max_tokens,
)

print(response.choices[0].message.content)

SELECT COUNT(*) AS total_bridges
FROM nbi_2025_ak;

Note: If there can be duplicate rows per bridge, use COUNT(DISTINCT STRUCTURE_NUMBER_008) instead.


In [18]:
content_prompt = """Tell me the whole query with COUNT(DISTINCT STRUCTURE_NUMBER_008)"""

In [19]:
response = client.chat.completions.create(
    model="gpt-5",
    messages=[{"role": "user", "content": content_prompt}],
    max_completion_tokens=max_tokens,
)

print(response.choices[0].message.content)

I don’t have your prior query, so here are ready-to-use patterns. Replace project.dataset.table and any grouping/filters as needed.

1) Overall distinct count
SELECT COUNT(DISTINCT STRUCTURE_NUMBER_008) AS bridge_count
FROM project.dataset.table;

2) Distinct count by state (example)
SELECT
  STATE_CODE_001 AS state,
  COUNT(DISTINCT STRUCTURE_NUMBER_008) AS bridge_count
FROM project.dataset.table
GROUP BY state
ORDER BY state;

3) Distinct count by state and county with a filter (portable syntax)
SELECT
  STATE_CODE_001 AS state,
  COUNTY_CODE_003 AS county,
  COUNT(DISTINCT CASE WHEN status = 'OPEN' THEN STRUCTURE_NUMBER_008 END) AS open_bridge_count
FROM project.dataset.table
WHERE year >= 2020
GROUP BY state, county
ORDER BY state, county;

4) If you prefer a subquery approach
SELECT COUNT(*) AS bridge_count
FROM (
  SELECT DISTINCT STRUCTURE_NUMBER_008
  FROM project.dataset.table
  WHERE year >= 2020
);

Notes:
- Keep STRUCTURE_NUMBER_008 as text if it has leading zeros; casting 

In [22]:
import pandas as pd
import duckdb as db

In [23]:
nbi_2025_ak = pd.read_csv("../data/nbi/AK25.csv")

In [25]:
query = """
SELECT COUNT(DISTINCT STRUCTURE_NUMBER_008) AS bridge_count
FROM nbi_2025_ak;
"""

db.sql(query)

┌──────────────┐
│ bridge_count │
│    int64     │
├──────────────┤
│         1224 │
└──────────────┘